# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [44]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier)
from sklearn.metrics import accuracy_score, classification_report


SEED = 8

In [22]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [23]:
spaceship.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 1.2+ MB


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [24]:
spaceship_df = spaceship.copy()

In [25]:
spaceship_df.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [26]:
spaceship_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 1.2+ MB


In [27]:
# Drop rows containing any missing (null) values
spaceship_df = spaceship_df.dropna()

# Verify that no missing values remain
print(spaceship_df.isnull().sum())

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64


In [28]:
# Split Cabin by '/' and take the first element (index 0)
spaceship_df['Cabin_Deck'] = spaceship_df['Cabin'].str.split('/').str[0]
# Check the extracted Deck categories
print(spaceship_df['Cabin_Deck'].value_counts(dropna=False))

Cabin_Deck
F    2152
G    1973
E     683
B     628
C     587
D     374
A     207
T       2
Name: count, dtype: int64


In [29]:
# Drop PassengerId and Name columns
spaceship_df = spaceship_df.drop(columns=['PassengerId', 'Name'])

# Verify remaining columns
print(spaceship_df.columns)

Index(['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP',
       'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Transported', 'Cabin_Deck'],
      dtype='str')


In [ ]:
# Create dummy / one-hot encoded columns for all non-numerical features
spaceship_df = pd.get_dummies(spaceship_df, drop_first=True)

# Convert all True/False boolean columns to integers (1 and 0)
spaceship_df = spaceship_df.astype(int)

# Verify updated columns and data types
print(spaceship_df.info())
print("\nSample rows:")
print(spaceship_df.head())

<class 'pandas.DataFrame'>
Index: 6606 entries, 0 to 8692
Columns: 5324 entries, Age to Cabin_Deck_T
dtypes: int64(5324)
memory usage: 268.4 MB
None

Sample rows:
   Age  RoomService  FoodCourt  ShoppingMall   Spa  VRDeck  Transported  \
0   39            0          0             0     0       0            0   
1   24          109          9            25   549      44            1   
2   58           43       3576             0  6715      49            0   
3   33            0       1283           371  3329     193            0   
4   16          303         70           151   565       2            1   

   HomePlanet_Europa  HomePlanet_Mars  CryoSleep_True  ...  \
0                  1                0               0  ...   
1                  0                0               0  ...   
2                  1                0               0  ...   
3                  1                0               0  ...   
4                  0                0               0  ...   

   Destinatio

**Perform Train Test Split**

In [33]:
# Separate features (X) and target (y)
X = spaceship_df.drop(columns=['Transported'])
y = spaceship_df['Transported']

# Split into 80% training set and 20% testing set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Check the dimensions of the split data
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

X_train shape: (5284, 5323)
X_test shape:  (1322, 5323)
y_train shape: (5284,)
y_test shape:  (1322,)


In [35]:
# Initialize and fit StandardScaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Transform the test data using the fitted scaler
X_test_scaled = scaler.transform(X_test)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [36]:
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=SEED),
    n_estimators=100,
    max_samples=1.0,
    bootstrap=True,
    random_state=SEED
)

bagging_model.fit(X_train_scaled, y_train)
print("Model Trained!")

Model Trained!


In [38]:
y_pred = bagging_model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print()
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Transported", "Transported"]
))

Accuracy: 79.20%

                 precision    recall  f1-score   support

Not Transported       0.79      0.79      0.79       656
    Transported       0.79      0.80      0.79       666

       accuracy                           0.79      1322
      macro avg       0.79      0.79      0.79      1322
   weighted avg       0.79      0.79      0.79      1322



In [39]:
pasting_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=SEED),
    n_estimators=100,
    max_samples=1.0,
    bootstrap=False,
    random_state=SEED
)

pasting_model.fit(X_train_scaled, y_train)
print("Model Trained!")

Model Trained!


In [40]:
y_pred_pasting = pasting_model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred_pasting) * 100:.2f}%")
print()
print(classification_report(
    y_test,
    y_pred_pasting,
    target_names=["Not Transported", "Transported"]
))

Accuracy: 77.16%

                 precision    recall  f1-score   support

Not Transported       0.77      0.76      0.77       656
    Transported       0.77      0.78      0.77       666

       accuracy                           0.77      1322
      macro avg       0.77      0.77      0.77      1322
   weighted avg       0.77      0.77      0.77      1322



- Random Forests

In [57]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=SEED,
    n_jobs=-1
)

random_forest_model.fit(X_train_scaled, y_train)
print("Model Trained!")

Model Trained!


In [58]:
# Predictions
y_pred_rf = random_forest_model.predict(X_test_scaled)

# Accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)

print(f"Random Forest Accuracy: {accuracy_rf * 100:.2f}%")
print()

# Classification report
print(classification_report(
    y_test,
    y_pred_rf,
    target_names=["Not Transported", "Transported"]
))

Random Forest Accuracy: 81.09%

                 precision    recall  f1-score   support

Not Transported       0.81      0.81      0.81       656
    Transported       0.81      0.82      0.81       666

       accuracy                           0.81      1322
      macro avg       0.81      0.81      0.81      1322
   weighted avg       0.81      0.81      0.81      1322



In [47]:
y_pred_rf_train = random_forest_model.predict(X_train_scaled)

accuracy_rf_train = accuracy_score(y_train, y_pred_rf_train)

print(f"Random Forest Train Accuracy: {accuracy_rf_train * 100:.2f}%")
print(f"Random Forest Test Accuracy: {accuracy_rf * 100:.2f}%")

Random Forest Train Accuracy: 99.98%
Random Forest Test Accuracy: 81.09%


- Gradient Boosting

In [48]:
gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=SEED
)

gradient_boosting_model.fit(X_train_scaled, y_train)
print("Gradient Boosting Model Trained!")

Gradient Boosting Model Trained!


In [49]:
# Predictions
y_pred_gb = gradient_boosting_model.predict(X_test_scaled)

# Accuracy
accuracy_gb = accuracy_score(y_test, y_pred_gb)

print(f"Gradient Boosting Accuracy: {accuracy_gb * 100:.2f}%")
print()

# Classification report
print(classification_report(
    y_test,
    y_pred_gb,
    target_names=["Not Transported", "Transported"]
))

Gradient Boosting Accuracy: 80.41%

                 precision    recall  f1-score   support

Not Transported       0.83      0.75      0.79       656
    Transported       0.78      0.85      0.81       666

       accuracy                           0.80      1322
      macro avg       0.81      0.80      0.80      1322
   weighted avg       0.81      0.80      0.80      1322



In [50]:
y_pred_gb_train = gradient_boosting_model.predict(X_train_scaled)

accuracy_gb_train = accuracy_score(y_train, y_pred_gb_train)

print(f"Gradient Boosting Train Accuracy: {accuracy_gb_train * 100:.2f}%")
print(f"Gradient Boosting Test Accuracy: {accuracy_gb * 100:.2f}%")

Gradient Boosting Train Accuracy: 81.68%
Gradient Boosting Test Accuracy: 80.41%


- Adaptive Boosting

In [51]:
adaboost_model = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=SEED
)

adaboost_model.fit(X_train_scaled, y_train)
print("AdaBoost Model Trained!")

AdaBoost Model Trained!


In [52]:
# Predictions
y_pred_ab = adaboost_model.predict(X_test_scaled)

# Accuracy
accuracy_ab = accuracy_score(y_test, y_pred_ab)

print(f"AdaBoost Accuracy: {accuracy_ab * 100:.2f}%")
print()

# Classification report
print(classification_report(
    y_test,
    y_pred_ab,
    target_names=["Not Transported", "Transported"]
))

AdaBoost Accuracy: 75.26%

                 precision    recall  f1-score   support

Not Transported       0.71      0.85      0.77       656
    Transported       0.82      0.65      0.73       666

       accuracy                           0.75      1322
      macro avg       0.76      0.75      0.75      1322
   weighted avg       0.76      0.75      0.75      1322



In [53]:
y_pred_ab_train = adaboost_model.predict(X_train_scaled)

accuracy_ab_train = accuracy_score(y_train, y_pred_ab_train)

print(f"AdaBoost Train Accuracy: {accuracy_ab_train * 100:.2f}%")
print(f"AdaBoost Test Accuracy: {accuracy_ab * 100:.2f}%")

AdaBoost Train Accuracy: 74.39%
AdaBoost Test Accuracy: 75.26%


Which model is the best and why?

In [54]:
results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Gradient Boosting",
        "AdaBoost"
    ],
    "Train Accuracy": [
        accuracy_rf_train,
        accuracy_gb_train,
        accuracy_ab_train
    ],
    "Test Accuracy": [
        accuracy_rf,
        accuracy_gb,
        accuracy_ab
    ]
})

results["Train Accuracy"] = results["Train Accuracy"] * 100
results["Test Accuracy"] = results["Test Accuracy"] * 100

results

,Model,Train Accuracy,Test Accuracy
0,Random Forest,99.981075,81.089259
1,Gradient Boosting,81.680545,80.408472
2,AdaBoost,74.394398,75.264750


## Ensemble Models – Results

Several ensemble methods were tested using the same feature engineering and preprocessing applied in the previous lab. The models evaluated were Bagging, Pasting, Random Forest, Gradient Boosting and AdaBoost.

Random Forest achieved the highest test accuracy at 81.09%, followed by Gradient Boosting with 80.41%. Bagging achieved 79.20%, Pasting 77.16%, and AdaBoost 75.26%.

Although Random Forest had the best test accuracy, it showed a significant difference between training accuracy (99.98%) and test accuracy (81.09%), which indicates a strong tendency towards overfitting. In comparison, Gradient Boosting had a much smaller difference between training (81.68%) and test accuracy (80.41%), suggesting better generalisation.

For the individual classes, Gradient Boosting achieved a recall of 0.85 for Transported, meaning it correctly identified 85% of the passengers who were actually transported. Random Forest had a more balanced performance, with precision, recall and F1-score of approximately 0.81–0.82 for both classes.

Overall, Random Forest produced the best test accuracy, while Gradient Boosting appears to be the more stable model with less overfitting. Therefore, both models are strong candidates, but Gradient Boosting may be preferable if generalisation and model stability are prioritised.